# Construye tu primer agente de IA con LangChain

**Taller práctico · EurusConf 2026**

En 90 minutos pasas de un modelo de lenguaje a un **agente que razona, usa herramientas y ejecuta tareas**. Tres prácticas, cada una sobre la anterior.

> **VS Code:** ya seguiste el README (entorno + `.env`).
> **Google Colab:** corre la celda de instalación y pega tu API key cuando te la pida.

## 0. Setup

In [ ]:
# Solo en Google Colab (en VS Code ya instalaste con requirements.txt):
%pip install -q "langchain>=1.0" langchain-google-genai langchain-openai python-dotenv pydantic

In [ ]:
# Carga tu API key y prepara el modelo (funciona en VS Code y en Colab)
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # lee .env si existe (VS Code)

PROVIDER = os.getenv("MODEL_PROVIDER", "google_genai")
MODEL_NAME = os.getenv("MODEL_NAME", "gemini-2.0-flash")

# En Colab (sin .env) pide la key de Gemini
if PROVIDER == "google_genai" and not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Pega tu GOOGLE_API_KEY: ")

def get_model():
    if PROVIDER == "openrouter":  # OpenRouter usa la API compatible con OpenAI
        return init_chat_model(f"openai:{MODEL_NAME}",
                               base_url="https://openrouter.ai/api/v1",
                               api_key=os.getenv("OPENROUTER_API_KEY"))
    return init_chat_model(f"{PROVIDER}:{MODEL_NAME}")

model = get_model()
print("Modelo listo:", PROVIDER, MODEL_NAME)

In [ ]:
# Smoke test: si esto responde, estás listo
print(model.invoke("Responde en una sola frase: ¿qué es LangChain?").content)

## Teoría mínima

- **LangChain** te da los bloques para construir con modelos: modelos, prompts, herramientas y agentes.
- Un **agente** = modelo + herramientas + un *loop*: recibe una tarea, decide si usar una herramienta, observa el resultado, repite, y responde.
- El ecosistema: **LangGraph** es el motor por debajo, **`create_agent`** (lo que usaremos hoy) es el harness estándar, **Deep Agents** es un harness más completo para tareas largas, y **LangSmith** sirve para observar y evaluar.

## Actividad 1 — Tu primera cadena

Una cadena conecta piezas con el operador `|`: `prompt | model | parser`.

In [ ]:
# DEMO: tu primera cadena con LCEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Explicas conceptos técnicos en una frase simple."),
    ("human", "{tema}"),
])
chain = prompt | model | StrOutputParser()
print(chain.invoke({"tema": "¿Qué es un agente de IA?"}))

### Tu turno

Pide una **salida estructurada** (datos validados, no texto suelto). Completa el modelo `Receta` y úsalo.

In [ ]:
from pydantic import BaseModel, Field

# TODO: agrega dos campos -> titulo (str) y duracion_min (int)
class Receta(BaseModel):
    pass  # TODO

# TODO: usa model.with_structured_output(Receta).invoke(...) para pedir una receta rápida
# resultado = ...
# print(resultado)

### Solución A1

In [ ]:
from pydantic import BaseModel, Field

class Receta(BaseModel):
    titulo: str = Field(description="nombre del platillo")
    duracion_min: int = Field(description="minutos para prepararlo")

resultado = model.with_structured_output(Receta).invoke("Dame una receta rápida de desayuno")
print(resultado)

## Actividad 2 — Tu primer agente

Le damos una herramienta al modelo y lo envolvemos con `create_agent`. El agente decide cuándo usarla; el loop la ejecuta solo.

In [ ]:
# DEMO: tu primer agente con una herramienta
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def buscar_precio(producto: str) -> str:
    """Devuelve el precio en dólares de un producto del catálogo."""
    catalogo = {"lampara": 25.0, "foco led": 8.5, "cable": 3.0}
    precio = catalogo.get(producto.lower())
    return f"{producto}: ${precio}" if precio is not None else f"No encontré '{producto}'"

agente = create_agent(
    model=model,
    tools=[buscar_precio],
    system_prompt="Eres un asistente de ventas. Usa las herramientas para responder con datos reales.",
)

r = agente.invoke({"messages": [{"role": "user", "content": "¿Cuánto cuesta una lampara?"}]})
print(r["messages"][-1].content)

### Tu turno

Crea una herramienta `horario_tienda` y arma un agente que use **dos** herramientas.

In [ ]:
# TODO: completa la herramienta para que devuelva el horario
@tool
def horario_tienda(dia: str) -> str:
    """Devuelve el horario de atención de la tienda."""
    pass  # TODO: devuelve un horario, ej. "Lunes a sábado: 9am-6pm"

# TODO: crea un agente con buscar_precio Y horario_tienda, y pregúntale:
# "¿a qué hora abren y cuánto cuesta un cable?"

### Solución A2

In [ ]:
@tool
def horario_tienda(dia: str) -> str:
    """Devuelve el horario de atención de la tienda."""
    return "Lunes a sábado: 9:00am - 6:00pm. Domingos cerrado."

mi_agente = create_agent(
    model=model,
    tools=[buscar_precio, horario_tienda],
    system_prompt="Eres un asistente de la tienda. Usa las herramientas.",
)
r = mi_agente.invoke({"messages": [{"role": "user", "content": "¿A qué hora abren y cuánto cuesta un cable?"}]})
print(r["messages"][-1].content)

## Actividad 3 — Tu agente completo

Varias herramientas que se **encadenan** para resolver un pedido de punta a punta.

In [ ]:
# DEMO: agente que encadena varias herramientas
@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática simple, ej. '25 * 3'."""
    try:
        return str(eval(expresion, {"__builtins__": {}}))
    except Exception as e:
        return f"Error: {e}"

@tool
def convertir_a_euros(monto_usd: float) -> str:
    """Convierte un monto de dólares a euros (tasa de ejemplo)."""
    return f"{monto_usd * 0.92:.2f} EUR"

asistente = create_agent(
    model=model,
    tools=[buscar_precio, calcular, convertir_a_euros],
    system_prompt="Eres un asistente de ventas. Encadena las herramientas para resolver el pedido.",
)
r = asistente.invoke({"messages": [{"role": "user", "content": "¿Cuánto costarían 3 lamparas en euros?"}]})
print(r["messages"][-1].content)

### Tu turno

Agrega una herramienta `aplicar_descuento(monto, porcentaje)` y pídele al agente:
*"dame el precio de 3 lámparas con 10% de descuento, en euros"*.

In [ ]:
# TODO: crea la herramienta de descuento y agrégala al agente
# @tool
# def aplicar_descuento(monto: float, porcentaje: float) -> str:
#     """Aplica un porcentaje de descuento a un monto."""
#     ...

### Solución A3

In [ ]:
@tool
def aplicar_descuento(monto: float, porcentaje: float) -> str:
    """Aplica un porcentaje de descuento a un monto."""
    return f"{monto * (1 - porcentaje / 100):.2f}"

asistente_pro = create_agent(
    model=model,
    tools=[buscar_precio, calcular, convertir_a_euros, aplicar_descuento],
    system_prompt="Eres un asistente de ventas. Encadena las herramientas para resolver el pedido.",
)
r = asistente_pro.invoke({"messages": [{"role": "user",
    "content": "Dame el precio de 3 lamparas con 10% de descuento, en euros."}]})
print(r["messages"][-1].content)

## Cierre

- Construiste una cadena y un agente con herramientas que resuelve un flujo real.
- **¿Cuándo usar qué?** `create_agent` para la mayoría; **LangGraph** cuando necesitas control total del flujo (estado, human-in-the-loop, multiagente); **Deep Agents** para tareas largas y complejas (planning, subagentes).
- **LangSmith** para observar y evaluar tus agentes en producción.

Recursos: https://docs.langchain.com · MART Automations: https://martautomations.com

## Reto extra (opcional) — Agente investigador web

Un agente que busca en internet y responde con información actual.

In [ ]:
# Requiere instalar la herramienta de búsqueda:
%pip install -q langchain-community ddgs
from langchain_community.tools import DuckDuckGoSearchRun

buscar_web = DuckDuckGoSearchRun()
investigador = create_agent(
    model=model,
    tools=[buscar_web],
    system_prompt="Eres un investigador. Busca en la web y responde con información actual.",
)
r = investigador.invoke({"messages": [{"role": "user", "content": "¿Qué novedades hay de LangChain?"}]})
print(r["messages"][-1].content)